In [ ]:
import os, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader

from unfairness.utils.config_loader import load_model_and_tokenizer
from unfairness.dataset import load_kb_struct
from unfairness.token import pad_sequences
from unfairness.infer import predict_with_rationales

CKPT    = "cv_test/torch/fold_1/best.ckpt" 
KB_DIR  = "local_database/KB"
MAX_LEN = 128
THRESH  = 0.5
TOP_K   = 3

class MiniDS(Dataset):
    def __init__(self, tok, texts, max_len):
        self.texts = list(texts)
        seqs = tok.texts_to_sequences(self.texts)
        self.ids = pad_sequences(seqs, maxlen=max_len, padding="post", truncating="post", value=0)
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        return {
            "input_ids": torch.tensor(self.ids[i], dtype=torch.long),
            "labels_multi": torch.zeros(5),
            "label_general": torch.tensor(0.0),
            "strong": {},
            "raw_text": self.texts[i],
        }

def collate(b):
    return {
        "input_ids": torch.stack([x["input_ids"] for x in b],0),
        "labels_multi": torch.stack([x["labels_multi"] for x in b],0),
        "label_general": torch.stack([x["label_general"] for x in b],0),
        "strong": [{} for _ in b],
        "raw_text": [x["raw_text"] for x in b],
    }

In [ ]:
lit, tok, kb_struct = load_model_and_tokenizer(
    ckpt_path=CKPT,
    max_len=MAX_LEN,
    kb_dir=KB_DIR,
    map_location="cuda"
)

In [ ]:
texts = [
    "We may share your data with affiliates for advertising purposes.",
    "Arbitration is mandatory unless prohibited by applicable law.",
]

dl = DataLoader(MiniDS(tok, texts, MAX_LEN), batch_size=8, shuffle=False, collate_fn=collate)
res = predict_with_rationales(
    lit_module=lit,
    dataloader=dl,
    kb_struct_or_texts=kb_struct,   # incluye Id/Tag/Explanation
    threshold=THRESH,
    top_k=TOP_K,
    return_for_all=True,
    use_scores=True,
)

# Pretty print
for r in res:
    print("TEXT:", r["text"])
    for cat, info in r["per_category"].items():
        if info["pred"] == 1 and info["rationales"]:
            top = info["rationales"][0]
            print(f"  {cat}  prob={info['prob']:.3f} → ({top['id']}, {top['tag']}) {top['text'][:100]} ...")
    print("-"*80)

In [ ]:
CSV_IN  = "my_texts.csv"                # debe existir
CSV_OUT = "my_texts_with_preds.csv"

df = pd.read_csv(CSV_IN)
dl = DataLoader(MiniDS(tok, df["text"].astype(str).tolist(), MAX_LEN), batch_size=64, shuffle=False, collate_fn=collate)

res = predict_with_rationales(
    lit_module=lit,
    dataloader=dl,
    kb_struct_or_texts=kb_struct,
    threshold=THRESH,
    top_k=TOP_K,
    return_for_all=False,
    use_scores=True,
)

rows = []
for r in res:
    base = {"text": r["text"], "general_pred": r["general_pred"], "general_prob": r["general_prob"]}
    for cat in ("A","CH","CR","LTD","TER"):
        pc = r["per_category"][cat]
        top = pc["rationales"][0] if pc["rationales"] else None
        base[f"pred_{cat}"] = pc["pred"]; base[f"prob_{cat}"] = pc["prob"]
        base[f"rat_{cat}_id"] = (top["id"] if top else "")
        base[f"rat_{cat}_tag"] = (top["tag"] if top else "")
        base[f"rat_{cat}_text"] = (top["text"] if top else "")
    rows.append(base)
pd.DataFrame(rows).to_csv(CSV_OUT, index=False)